<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c30-header.png" alt="Nextia Learning · Testing AI Applications: Manual and AI-Assisted QA" width="100%">

# Solution: Agent-ready reports from real open-source issues

**[Agent-ready reports from real open-source issues](https://learning.nextia-ai.com/courses/ai-qa/m08/agent-ready-reports-from-open-source-issues/)** · Testing AI Applications: Manual and AI-Assisted QA · Module 8, case study 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** rate 30 real issues from open-source projects for agent-readiness, check each one against the fix and the test that were merged for it, compare ways to decide which issues can go to a coding agent as written, rewrite a weak issue in the agent-ready format, and check the rewrite against the real test.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in a few seconds. |
| **Needs** | An internet connection for the setup cell. No account and no key: the model ratings were recorded once, and the notebook replays them. Standard library only. One optional part downloads Django (about 10 MB) to run a real test. |
| **You should know** | [Agent-ready issue templates](https://learning.nextia-ai.com/courses/ai-qa/m05/agent-ready-issue-templates/), [Turn a bug into a failing case](https://learning.nextia-ai.com/courses/ai-qa/m05/turn-a-bug-into-a-failing-case/) and [Verify the agent's fix](https://learning.nextia-ai.com/courses/ai-qa/m05/verify-the-agents-fix/). |
| **Data** | SWE-bench Lite (Jimenez et al., 2024), MIT licence: real issues from 12 open-source Python projects, each with its merged fix and test. |
| **Tested** | Python 3.12 and 3.13 with the standard library only, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ai-qa/m08/agent-ready-reports-from-open-source-issues/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C30/M08-L02-agent-ready-reports-from-open-source-issues/agent-ready-reports-from-open-source-issues-solution.ipynb).

## Setup: get the data

Run the next cell. It downloads the case study's 9 small files (about 0.2 MB in all) from the Nextia labs repository into a folder `agent-ready-issues/data`, and checks each checksum. It needs an internet connection and no account. If a `data` folder is next to this notebook, the cell copies the files from there.

You should see `Ready: 9 files, checksums OK.`

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `agent-ready-issues` (in Colab: the **Files** panel on the left), then run the cell again.

Data: SWE-bench Lite (Jimenez et al., 2024, arXiv:2310.06770), https://huggingface.co/datasets/SWE-bench/SWE-bench_Lite, MIT licence. The issue texts, fixes and tests come from 12 open-source Python projects on GitHub. Nextia Learning selected 31 issues and kept some fields. The ratings and checks are by the course author; the model ratings are recorded outputs of a hosted model.

In [ ]:
# Setup: download the data files into agent-ready-issues/data and check them. Standard library only.
import csv, hashlib, json, os, re, shutil, subprocess, urllib.request
from collections import Counter
from pathlib import Path
from statistics import median

URL = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C30/M08-L02-agent-ready-reports-from-open-source-issues/data/"
FILES = {
    "issues_sample.jsonl": "0aabeaa989f768f1367c7aedb65bdf321840fd7bc39f892324d8e2549c2933d7",
    "author_ratings.csv": "87d0e347b36ffbb718e2c47320b36b1e91f790b261fd3c4bc40d9a4ae921e9dd",
    "author_checks.csv": "8be337de985bb4e3d74fdc1e88b1868b58d2105df6aff766e4ab6d253ca2b424",
    "model_ratings.csv": "015def0ee9df70ade16be71aed3d03df65cbfea9ce2a99b35984754d7996970a",
    "rewrites.jsonl": "039c838c9a4b899069733e6ec90b03dcf2c9b0ee22c744adc0e66ea88d9438e2",
    "rewrite_ratings.csv": "cb86033a099ba4342cd70ef6f415193dee19516cf2b2edadff3f3c4c5f35f301",
    "test_rewrite_check.py": "86c3d48defb17618a742955a4d096baeb11504ff829a116c6746d0562c684fca",
    "check_rewrite_django-15213.txt": "b818829329c379fc4fd999cfb81269a5cd7f27b87ef1440608ee14ac951d2f9d",
    "LICENSE-SWE-bench.txt": "2bd2e08df7147f67a69b42c10efae09bd4bf119df397371036187d5dd1b02f57",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("agent-ready-issues").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for name, expected in FILES.items():
    target = WORK / "data" / name
    if target.exists() and sha256(target) == expected:
        continue
    if (HERE / "data" / name).exists():
        shutil.copy(HERE / "data" / name, target)
    else:
        try:
            urllib.request.urlretrieve(URL + name, target)
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(target), URL + name]).returncode:
                raise SystemExit(f"Could not download {URL + name}. Check your internet connection.")
    if sha256(target) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder agent-ready-issues and run again.")
os.chdir(WORK)
print(f"Ready: {len(FILES)} files, checksums OK.")

## Setup: helpers

Run the next cell. It defines the rubric, small functions to load the data and read a diff, and the checks for your tasks. You should see no output.

In [ ]:
# Helpers: the rubric, loading the data, reading a diff, and checks that hide the answer.
CRITERIA = ["reproduction", "versions", "expected", "acceptance", "scope"]
RUBRIC = {  # 2 / 1 / 0 for each criterion, from the course's agent-ready template
    "reproduction": ("runs as written, with the actual output", "something to start from, but you must guess", "none"),
    "versions": ("library version or commit, and the environment that matters", "partly, or vague", "none"),
    "expected": ("an exact value or a clear Must / Must not rule", "implied, vague or undecided", "not stated"),
    "acceptance": ("a done condition beyond the one example", "only the one example", "nothing to check"),
    "scope": ("the suspected cause or code place, or limits", "only the feature or API", "only the symptom"),
}

def read_jsonl(name):
    return [json.loads(line) for line in open(f"data/{name}", encoding="utf-8")]

def read_csv(name, key="instance_id"):
    """A CSV file as {key: row}; the five criteria become numbers."""
    rows = {}
    for row in csv.DictReader(open(f"data/{name}", encoding="utf-8")):
        for c in CRITERIA:
            if c in row:
                row[c] = int(row[c])
        rows[row[key]] = row
    return rows

def total(row):
    return sum(row[c] for c in CRITERIA)

def show(text, max_lines=30):
    """Print a long text, cut after max_lines lines."""
    lines = text.strip().splitlines()
    print("\n".join(lines[:max_lines]) + (f"\n[... {len(lines) - max_lines} more lines]" if len(lines) > max_lines else ""))

def added_lines(diff):
    """The lines a diff adds, without the leading +."""
    return [l[1:] for l in diff.splitlines() if l.startswith("+") and not l.startswith("+++")]

def place_names(patch):
    """Names that point to the fix: changed file names, and the functions and classes in the diff."""
    names = {Path(p).stem for p in re.findall(r"^diff --git a/(\S+) b/", patch, flags=re.M)}
    names |= set(re.findall(r"^@@[^@]*@@\s*(?:async\s+)?(?:def|class)\s+(\w+)", patch, flags=re.M))
    names |= set(re.findall(r"^[ +-]\s*(?:async\s+)?(?:def|class)\s+(\w+)", patch, flags=re.M))
    return sorted(n for n in names if n not in {"__init__", "tests", "test"} and len(n) > 3)

def names_in_text(text, names):
    return [n for n in names if re.search(rf"(?<![\w]){re.escape(n)}(?![\w])", text)]

def triage(flags, ready):
    """flags[i] True = send issue i to the agent as written. Count the two kinds of misrouted issue."""
    sent_not_ready = sum(f and not r for f, r in zip(flags, ready))
    held_ready = sum(r and not f for f, r in zip(flags, ready))
    return {"sent": sum(flags), "sent_ready": sum(f and r for f, r in zip(flags, ready)),
            "sent_not_ready": sent_not_ready, "held_ready": held_ready, "misrouted": sent_not_ready + held_ready}

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()[:16]

def expect(what, yours, expected):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Amara is the QA lead of a team that maintains open-source Python libraries. Her manager, Jonas, wants to hand open issues to a coding agent as they are written. Amara wants evidence first: how often does an issue, as written, say enough for a fixer to make the change that the maintainers accepted? And can a short rubric tell the ready issues from the others?

The [lesson page](https://learning.nextia-ai.com/courses/ai-qa/m08/agent-ready-reports-from-open-source-issues/) tells the full story. This notebook runs every step.

## 2. The data

Each line of `issues_sample.jsonl` is one real issue from SWE-bench Lite: the issue text, the fix that was merged for it (a diff), and the test that came with the fix. Load the sample and count it.

In [ ]:
issues = read_jsonl("issues_sample.jsonl")
by_id = {i["instance_id"]: i for i in issues}
sample = [i for i in issues if i["role"] == "sample"]
new_input = next(i for i in issues if i["role"] == "new-input")
print(len(sample), "issues to rate, 1 kept apart:", new_input["instance_id"])
print(Counter(i["repo"] for i in sample).most_common())
print("words per issue text, median:", median(len(i["problem_statement"].split()) for i in sample))
print("fields:", sorted(sample[0]))

> **Check.** You should see 30 issues from 12 repositories, with a median of 120.5 words per text.

## 3. The rubric on one issue

The rubric has five criteria from the course's agent-ready template. Each scores 0, 1 or 2 **from the issue text alone**. Print it, then read one issue.

In [ ]:
for c in CRITERIA:
    print(f"{c:13} 2 = {RUBRIC[c][0]}; 1 = {RUBRIC[c][1]}; 0 = {RUBRIC[c][2]}")

Read this Django issue. It is short.

In [ ]:
show(by_id["django__django-15851"]["problem_statement"])

> **Predict.** Before you run the next cell, give each criterion a score from 0 to 2. Write your five scores down.

The next cell shows the course author's scores and reasons.

In [ ]:
author = read_csv("author_ratings.csv")
row = author["django__django-15851"]
print({c: row[c] for c in CRITERIA}, "total", total(row))
print(row["note"])

> **Check.** The author gave 8 of 10: full marks for reproduction, expected behaviour and scope; 1 for versions (psql 10.21, but no Django version); 1 for acceptance (only the one example).

## 4. All 30 issues, from the text alone

The author rated every issue before opening any fix or test. Count how often each criterion got 0, 1 and 2.

In [ ]:
ids = [i["instance_id"] for i in sample]
for c in CRITERIA:
    counts = Counter(author[i][c] for i in ids)
    print(f"{c:13} 2: {counts[2]:2}   1: {counts[1]:2}   0: {counts[0]:2}")
totals = [total(author[i]) for i in ids]
print("total: median", median(totals), "| 7 or more:", sum(t >= 7 for t in totals), "of", len(ids))

> **Check.** Versions are missing in 16 of 30 texts, and only 2 texts give a done condition beyond their one example.

## 5. The answer key: the merged fix and test

Now open what the maintainers merged. For each issue, the author answered two questions: does the text point to the place of the fix? Does the merged test check what the text expects? First, one issue where the answer is "partly": the text proposes `mode="b"`.

In [ ]:
flask = by_id["pallets__flask-4992"]
print([line for line in flask["problem_statement"].splitlines() if "mode=" in line])
print([line.strip() for line in added_lines(flask["test_patch"]) if "from_file" in line or "text=" in line])

> **Check.** The text asks for `mode="b"`. The merged test calls `from_file(..., tomllib.load, text=False)`. A fix that follows the text exactly fails the test that the maintainers wrote.

Count the answers for all 30 issues, and compare the rubric totals of the three groups.

In [ ]:
checks = {r["instance_id"]: r for r in csv.DictReader(open("data/author_checks.csv", encoding="utf-8"))}
print("does the test check what the text expects?", Counter(checks[i]["test_matches"] for i in ids))
print("does the text point to the fix?          ", Counter(checks[i]["points_to_fix"] for i in ids))
for group in ("yes", "partly", "no"):
    group_totals = sorted(total(author[i]) for i in ids if checks[i]["test_matches"] == group)
    print(f"{group:6} {len(group_totals):2} issues, rubric totals {group_totals}")

> **Check.** The test matches the text on 10 issues, partly on 16, and not on 4. The totals rise from "no" to "yes", but they overlap.

A computer can check one part of "points to the fix": does the text name a file, function or class that the merged diff changes?

In [ ]:
named = {i["instance_id"]: names_in_text(i["problem_statement"], place_names(i["patch"])) for i in sample}
print(Counter(f"{checks[i]['points_to_fix']} / {'named' if named[i] else 'not named'}" for i in ids))
print("pytest-dev__pytest-7432 names:", named["pytest-dev__pytest-7432"])

## 6. Which issues can go to the agent as written?

Call an issue **ready** when the merged test checks what its text expects (`yes`). A triage rule sends some issues to the agent as written and holds the others for a rewrite. One measure for every rule: the number of **misrouted** issues, sent but not ready, plus ready but held.

In [ ]:
ready = [checks[i]["test_matches"] == "yes" for i in ids]
words = [len(by_id[i]["problem_statement"].split()) for i in ids]
rules = {
    "send everything": [True] * 30,
    "rewrite everything first": [False] * 30,
    "has code or a traceback": [any(s in by_id[i]["problem_statement"] for s in ("```", ">>>", "Traceback")) for i in ids],
    "long text (above the median)": [w > median(words) for w in words],
    "rubric total 7+": [total(author[i]) >= 7 for i in ids],
    "expected = 2": [author[i]["expected"] == 2 for i in ids],
}
for name, flags in rules.items():
    r = triage(flags, ready)
    print(f"{name:29} sent {r['sent']:2} ({r['sent_ready']:2} ready) | misrouted {r['misrouted']:2}")

> **Check.** The rubric total of 7 or more misroutes 7 issues; "send everything" misroutes 20, and "rewrite everything first" 10.

> **Predict.** What happens to the number of misrouted issues if the bar moves from 7 to 6, or to 8? Write it down, then run the next cell.

In [ ]:
for bar in range(3, 11):
    r = triage([total(author[i]) >= bar for i in ids], ready)
    print(f"total {bar}+: sent {r['sent']:2}, misrouted {r['misrouted']:2} (sent not ready {r['sent_not_ready']:2}, ready held {r['held_ready']:2})")

### Change one thing: the cost of each mistake

Equal weights treat a wasted agent run like a few minutes of rewriting. If a wasted run (a wrong fix, a review, a reopened issue) costs **3** and an unneeded rewrite costs **1**, the order changes. Run the same rules with these weights.

In [ ]:
for name, flags in rules.items():
    r = triage(flags, ready)
    print(f"{name:29} weighted cost {3 * r['sent_not_ready'] + 1 * r['held_ready']:3}")

> **Check.** With these weights, "rewrite everything first" (10) is cheaper than the rubric total (17). The page explains what this means for the recommendation.

## 7. A model rates the same issues

A small hosted model rated every issue with the same rubric, three times, from the text alone. The ratings were recorded once; here you replay them. Compare the model's median with the author's score, criterion by criterion.

In [ ]:
model = {}
for row in csv.DictReader(open("data/model_ratings.csv", encoding="utf-8")):
    model.setdefault(row["instance_id"], []).append({c: int(row[c]) for c in CRITERIA})
model_median = {i: {c: int(median(run[c] for run in runs)) for c in CRITERIA} for i, runs in model.items()}
for c in CRITERIA:
    same = sum(model_median[i][c] == author[i][c] for i in ids)
    lower = sum(model_median[i][c] < author[i][c] for i in ids)
    stable = sum(len({run[c] for run in model[i]}) == 1 for i in ids)
    print(f"{c:13} same as the author {same:2}/30, model lower {lower}, same in all 3 runs {stable:2}/30")
changed = sum(len({total(run) for run in model[i]}) > 1 for i in ids)
print("issues whose model total changed between runs:", changed)
r = triage([total(model_median[i]) >= 7 for i in ids], ready)
print("model rubric total 7+: sent", r["sent"], "| misrouted", r["misrouted"])

> **Check.** The model agrees on 21 to 26 of 30 per criterion. It is stricter on reproduction (lower on 8 issues), and its total changes between runs on 10 issues.

## 8. Rewrite a weak issue

Here is the original Django issue `django__django-15213` (author total 5) and the author's agent-ready rewrite. Read both.

In [ ]:
show(by_id["django__django-15213"]["problem_statement"], 20)
print("\n" + "=" * 70 + "\n")
rewrites = {r["instance_id"]: r for r in read_jsonl("rewrites.jsonl")}
show(rewrites["django__django-15213"]["problem_statement"], 60)

The author and the model rated the three rewrites with the same rubric. Compare before and after.

In [ ]:
after = read_csv("rewrite_ratings.csv")
for i in after:
    print(f"{i:26} author {total(author[i])} -> {total(after[i])}")

## 9. Check the rewrite against the merged fix and test

The rewrite's acceptance criteria became a failing case with three tests, `data/test_rewrite_check.py`. Print it. The author ran it on Django at the issue's base commit, then with the merged fix. The recorded output is in `data/check_rewrite_django-15213.txt`. Then see the tests that the maintainers merged with the fix.

In [ ]:
show(open("data/test_rewrite_check.py", encoding="utf-8").read(), 40)
print("\n" + "=" * 70 + "\n")
merged = added_lines(by_id["django__django-15213"]["test_patch"])
print("\n".join(line for line in merged if "def test" in line or "assert" in line))

Now read the recorded run: the SQL before and after the fix, and the results of both sets of tests.

In [ ]:
run = open("data/check_rewrite_django-15213.txt", encoding="utf-8").read()
print("\n".join(line for line in run.splitlines() if line.startswith(("###", "Q(", "~Q(", "Ran ", "OK", "FAILED", "exit"))))

> **Check.** Before the fix, the rewrite's case and the merged tests both fail with the same error. After the fix, both pass, and so does the whole `annotations` suite (80 tests). The merged tests add two cases that the rewrite did not name: the annotation inside `Count()` and `Sum()`.

### Optional: run the check yourself

This part downloads Django at the issue's base commit (about 10 MB) and installs two small packages. It needs Python 3.10 to 3.12 (Colab and Kaggle have one of these) and the `patch` program (Colab and Kaggle have it; on Windows, use WSL). It takes about one minute. Set `RUN_DJANGO = True` to run it.

In [ ]:
RUN_DJANGO = False  # set to True to run the real check
DJANGO = "https://codeload.github.com/django/django/tar.gz/03cadb912c78b769d6bf4a943a2a35fc1d952960"
if RUN_DJANGO:
    import sys, tarfile
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asgiref==3.4.1", "sqlparse==0.4.2"], check=True)
    urllib.request.urlretrieve(DJANGO, "django.tar.gz")
    shutil.rmtree("django-src", ignore_errors=True)  # start from a clean copy on every run
    tarfile.open("django.tar.gz").extractall("django-src", filter="data")
    src = next(Path("django-src").iterdir()).resolve()
    shutil.copy("data/test_rewrite_check.py", src / "tests/annotations/test_rewrite_check.py")  # the rewrite's failing case
    def runtests(*labels):
        out = subprocess.run([sys.executable, "runtests.py", "--parallel", "1", *labels], cwd=src / "tests",
                             env=dict(os.environ, PYTHONPATH=str(src)), capture_output=True, text=True)
        lines = [l for l in out.stderr.splitlines() if l.startswith(("Ran ", "OK", "FAILED", "django.db.utils"))]
        print(*(lines or out.stderr.splitlines()[-3:]), sep="\n")
    issue = by_id["django__django-15213"]
    print("1. the rewrite's case, base commit:"); runtests("annotations.test_rewrite_check")
    subprocess.run(["patch", "-p1", "--quiet"], cwd=src, input=issue["test_patch"], text=True, check=True)
    subprocess.run(["patch", "-p1", "--quiet"], cwd=src, input=issue["patch"], text=True, check=True)
    print("2. the rewrite's case, after the merged fix:"); runtests("annotations.test_rewrite_check")
    print("3. the annotations suite with the merged tests, after the fix:"); runtests("annotations")
else:
    print("Skipped. Set RUN_DJANGO = True to run the real check.")

## 10. End to end: a new issue

One more Django issue was kept apart from the start. Run the whole method on it: read it, rate it, decide, then open the answer key.

In [ ]:
show(new_input["problem_statement"])

> **Your turn.** Rate the new issue from its text alone. Replace each `...` with 0, 1 or 2, then run the cell. The check tells you on how many criteria you agree with the course author, not the author's scores.

In [ ]:
my_rating = {"reproduction": 2, "versions": 0, "expected": 2, "acceptance": 1, "scope": 1}

In [ ]:
if any(v not in (0, 1, 2) for v in my_rating.values()):
    print("Not yet: every score must be 0, 1 or 2.")
else:
    same = sum(my_rating[c] == author[new_input["instance_id"]][c] for c in CRITERIA)
    print(f"Check passed: you agree with the course author on {same} of 5 criteria. Differences of one point are normal; read the page for the author's reasons.")

Now the decision and the answer key: the author's and the model's totals, the rule's decision, and what the merged test checks.

In [ ]:
nid = new_input["instance_id"]
print("author total", total(author[nid]), "| model total", total(model_median[nid]),
      "| decision with the bar of 7:", "send" if total(author[nid]) >= 7 else "rewrite first")
print("merged test:", [l.strip() for l in added_lines(new_input["test_patch"]) if "memoryview" in l or "assert" in l])
print("names in the text that the fix changes:", names_in_text(new_input["problem_statement"], place_names(new_input["patch"])))

> **Check.** Both raters give 6, so the rule says "rewrite first". Yet the merged test checks exactly what the text expects: `HttpResponse(memoryview(b'memoryview')).content == b'memoryview'`. The missing version cost a rewrite that the agent did not need. The page discusses this.

## 11. Tasks

> **Your turn.** How many of the 30 texts give **no version at all** (versions = 0)? Use `author` and `ids`.

In [ ]:
no_version = sum(author[i]["versions"] == 0 for i in ids)

In [ ]:
expect("the number of texts with no version", no_version, 16)

> **Your turn.** A colleague proposes a simpler rule: send an issue when its **reproduction** score is 2. Compute its result with `triage()` and `ready`, and keep the whole dictionary.

In [ ]:
repro_rule = triage([author[i]["reproduction"] == 2 for i in ids], ready)

In [ ]:
expect_hash("the result of the reproduction rule", repro_rule, "9dcaa341fe7c5c5c")
if isinstance(repro_rule, dict) and "misrouted" in repro_rule:
    print("Compare its misrouted count with the rubric total's 7: a runnable example alone does not make an issue ready.")

> **Your turn.** For the new issue, which merged test function was added? Find the name in `new_input["test_patch"]` (a line that starts with `def test`), and write it as a string without `def ` and without the brackets.

In [ ]:
new_test_name = next(l.strip().split("(")[0].removeprefix("def ") for l in added_lines(new_input["test_patch"]) if l.strip().startswith("def test"))

In [ ]:
expect_hash("the name of the merged test", new_test_name, "39baa8e6f56f187b")

> **Your turn.** Write one acceptance criterion for the new issue as a Must rule, in one sentence, in the variable below. There is no automatic check: compare it with the merged test that you found, as the page does for the rewrite. Does your rule check the same thing? Does it name a neighbour that the merged test does not?

In [ ]:
my_acceptance = ("Must: HttpResponse(memoryview(b'My Content')).content == b'My Content', and str and bytes "
                 "content keep their current results.")
print(my_acceptance)

## Recap

- From the text alone, 13 of 30 real issues reached 7 of 10 on the rubric. Versions and acceptance criteria were the parts most often missing.
- The merged test checked what the text expects on only 10 of 30 issues. On 16, it checked more or something different: a parameter name, an exact format, an error type.
- As one measure, the rubric total of 7 or more misrouted 7 issues, against 20 for "send everything" and 10 for "rewrite everything first". When a wasted agent run costs more than a rewrite, rewriting everything first was cheapest.
- A model applied the rubric close to the author, but more strictly on reproduction, and not the same on every run.
- A rewrite with Must rules and a failing case matched the core of the merged test, and both failed before the fix and passed after it. The maintainers' test added neighbours the rewrite had not named.

Read the [lesson page](https://learning.nextia-ai.com/courses/ai-qa/m08/agent-ready-reports-from-open-source-issues/) for the recommendation, the limits and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [A safety test pass with open attack sets](https://learning.nextia-ai.com/courses/ai-qa/m08/safety-test-pass-with-open-attack-sets/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ai-qa/m08/agent-ready-reports-from-open-source-issues/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The issues, fixes and tests are real, from open-source projects (SWE-bench Lite, MIT).